In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/NLP/roberta_model"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.eval()

In [ ]:
import torch

text = "This campaign used misinformation to influence voters"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**inputs)

pred_id = torch.argmax(outputs.logits, dim=1).item()
pred_label = model.config.id2label[pred_id]

print("Prediction:", pred_label)

In [ ]:
# Install once
!pip install transformers datasets torch -q

import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# =========================
# 1) Load dataset
# =========================
df2 = pd.read_csv("adversary_actions_dataset.csv", engine="python", on_bad_lines="skip")

df2["justification"] = df2["justification"].fillna("")
df2["text"] = df2["statement"] + " " + df2["justification"]

# =========================
# 2) Encode labels
# =========================
labels_unique = sorted(df2["label"].unique())
label2id = {label: i for i, label in enumerate(labels_unique)}
id2label = {i: label for label, i in label2id.items()}

df2["labels"] = df2["label"].map(label2id)

# =========================
# 3) Train/test split
# =========================
train_df, test_df = train_test_split(
    df2[["text", "labels", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df2["label"]
)

# =========================
# 4) Convert to HuggingFace Dataset
# =========================
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

# =========================
# 5) Tokenizer
# =========================
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# =========================
# 6) Load model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 7) Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

# =========================
# 8) Training setup
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# =========================
# 9) Train
# =========================
trainer.train()

# =========================
# 10) Evaluate
# =========================
results = trainer.evaluate()
print("\nEvaluation results:")
print(results)

# =========================
# 11) Detailed classification report
# =========================
pred_output = trainer.predict(test_dataset)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
))

In [ ]:
!pip install transformers datasets torch scikit-learn pandas -q

import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

drive.mount('/content/drive')

# =========================
# 1) Define paths
# =========================
DATA_PATH = "/content/countermeasures_dataset.csv"
SAVE_PATH = "/content/drive/MyDrive/NLP/countermeasures_roberta_model"

# =========================
# 2) Load dataset
# =========================
df = pd.read_csv(DATA_PATH, engine="python", on_bad_lines="skip")

df["justification"] = df["justification"].fillna("")
df["text"] = df["statement"] + " " + df["justification"]

# =========================
# 3) Encode labels
# =========================
labels_unique = sorted(df["label"].unique())
label2id = {label: i for i, label in enumerate(labels_unique)}
id2label = {i: label for label, i in label2id.items()}

df["labels"] = df["label"].map(label2id)

# =========================
# 4) Train/test split
# =========================
train_df, test_df = train_test_split(
    df[["text", "labels", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# =========================
# 5) Convert to HuggingFace Dataset
# =========================
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

# =========================
# 6) Tokenizer
# =========================
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# =========================
# 7) Load model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# =========================
# 8) Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

# =========================
# 9) Training setup
# =========================
training_args = TrainingArguments(
    output_dir="./results_countermeasures",
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs_countermeasures"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# =========================
# 10) Train
# =========================
trainer.train()

# =========================
# 11) Evaluate
# =========================
results = trainer.evaluate()
print("\nEvaluation results:")
print(results)

# =========================
# 12) Detailed classification report
# =========================
pred_output = trainer.predict(test_dataset)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print("\nClassification Report:\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))]
))

# =========================
# 13) Save model and tokenizer
# =========================
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"\nModel and tokenizer saved to: {SAVE_PATH}")

In [ ]:
trainer.save_model("roberta_model")
tokenizer.save_pretrained("roberta_model")